# 4. Evaluation

Compare models, run error analysis, and summarize forecast performance for the business report.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from bsdf.config import get_config
from bsdf.data import load_processed_data, split_time_ordered_data
from bsdf.evaluation import build_prediction_frame, summarize_error_by_hour, summarize_forecast_by_day
from bsdf.features import add_time_features, create_feature_target_split
from bsdf.utils import load_model, save_table

CONFIG = get_config(PROJECT_ROOT)

In [2]:
comparison = pd.read_csv(CONFIG.processed_dir / "model_comparison.csv")
data = add_time_features(load_processed_data(CONFIG))
train_data, test_data = split_time_ordered_data(data, CONFIG.test_fraction)
X_test, y_test = create_feature_target_split(test_data)

best_model = load_model(CONFIG.model_dir / "best_model.joblib")
prediction_frame = build_prediction_frame(best_model, test_data, X_test)
save_table(prediction_frame, CONFIG.processed_dir / "hourly_predictions_test.csv")

comparison

,model,mean_absolute_deviation,mean_absolute_percentage_error,r2_score
0,xgboost,50.905449,0.490943,0.886153
1,random_forest,53.471518,0.354383,0.868960
2,gradient_boosting,57.002632,0.608443,0.860422
3,linear_regression,98.690537,2.113512,0.631634


In [3]:
hourly_error = summarize_error_by_hour(prediction_frame)
daily_forecast = summarize_forecast_by_day(prediction_frame)
save_table(hourly_error, CONFIG.processed_dir / "error_by_hour.csv")
save_table(daily_forecast, CONFIG.processed_dir / "daily_forecast_analysis.csv")

hourly_error.head(24)

,hr,actual_mean,prediction_mean,mad
0,0,69.29,62.000000,23.26
1,1,43.60,39.240002,18.49
2,2,28.28,29.610001,17.99
3,3,14.13,17.820000,10.73
4,4,8.63,12.110000,5.28
5,5,28.60,26.350000,11.33
6,6,101.22,87.949997,26.40
7,7,286.36,245.179993,82.02
8,8,485.34,421.369995,113.77
9,9,291.43,254.779999,61.54


In [4]:
daily_forecast.tail(10)

,date,actual_total,predicted_total,mad
137,2012-12-22,1749,2875.020020,47.79
138,2012-12-23,1787,2802.729980,45.69
139,2012-12-24,920,2463.929932,72.09
140,2012-12-25,1013,2804.810059,77.90
141,2012-12-26,441,1676.869995,51.49
142,2012-12-27,2114,3352.620117,53.20
143,2012-12-28,3095,3494.800049,47.76
144,2012-12-29,1341,2251.469971,37.94
145,2012-12-30,1796,2521.830078,32.20
146,2012-12-31,2729,3232.310059,76.37


## Short Analysis Report

The hourly bike-sharing demand is highly structured by time. Working days show commute peaks, while non-working days spread rentals across daytime hours. Weather, temperature, humidity, and season explain additional variation. The target distribution is right-skewed, meaning planners need a model that handles both many low-demand hours and a smaller number of operationally important peak hours.

For a daily planning service, the selected business model is the best model from the full comparison of Linear Regression, Random Forest, Gradient Boosting, and XGBoost. In the executed workflow, XGBoost achieved the lowest mean absolute deviation at 50.91 hourly rentals. This metric represents the average hourly bike count error and is easy for planners to translate into operational buffer decisions. The model is packaged as a single pipeline with preprocessing, so colleagues can retrain, version, and load it consistently.

For production, the service should refresh forecasts daily after the latest completed demand data is available. Intraday refreshes are useful when weather changes materially or live demand departs from plan. This model is strongest for short- to medium-term operational planning where calendar and weather assumptions are credible; long-range annual planning should include broader business signals such as fleet changes, station changes, pricing, holidays, and local events.